# 
RLCT Estimation of Sorting

This Jupyter Notebook aims to measure the Real Log Canonical Threshold (RLCT) for a small 3-layer transformer model (~280,000 parameters) trained to sort sequences of 20 digits consisting of the numbers 0-19. It uses both Stochastic Gradient Nose-Hoover Thermostat (SGNHT) and Stochastic Gradient Langevin Dynamics (SGLD) as sampling methods.

## Main Steps:

1. **Data Preparation**: Generate the dataset of numbers to sort.
2. **Model Training**: Train a transformer model using stochastic gradient descent.
3. **Model Evaluation**: Evaluate the model's performance on a test set.
4. **RLCT Estimation**: Use SGNHT and SGLD samplers to estimate RLCT.
5. **Plotting**: Visualize train and test losses, and RLCT estimates.

In [1]:
import itertools
from tqdm.notebook import tqdm
import numpy as np
from math import comb
import torch

def generate_combinations(n, k):
    """
    Generate all combinations of n choose k, excluding specific combinations.
    
    Parameters:
    n (int): Total number of items.
    k (int): Number of items to choose.
    exclude_set (set of tuples): Combinations to exclude.
    
    Yields:
    tuple: The next valid combination.
    """
    def comb_descending(combination):
        return all(combination[i] >= combination[i + 1] for i in range(len(combination) - 1))
        
    for combination in tqdm(itertools.combinations(range(n - 1, 0, -1), k)):
        #if comb_descending(combination):
        yield combination

def swap(arr, idx, jdx):
    temp = arr[idx]
    arr[idx] = arr[jdx]
    arr[jdx] = temp

def create_sequence(arr, indices):
    new_arr = arr.copy()
    for idx in indices:
        swap(new_arr, idx, idx - 1)
    return new_arr

def create_sequences(arr, combinations):
    sequences = []
    for combination in combinations:
        sequences.append(create_sequence(arr, list(combination)))
    return sequences

def apply_swaps(arr, num_swaps):
    for idx in range(len(arr) - 1):
        if arr[idx] > arr[idx + 1]:
            temp = arr[idx]
            arr[idx] = arr[idx + 1]
            arr[idx + 1] = temp
            num_swaps -= 1
        if num_swaps == 0:
            return arr, True
    return arr, False

def count_inversions(arr):
    """
    Count the number of inversions (adjacent swaps) needed to sort the array.

    Parameters:
    arr (list): List of integers to count inversions in.

    Returns:
    int: Number of inversions.
    """
    if len(arr) < 2:
        return 0

    mid = len(arr) // 2
    left = arr[:mid]
    right = arr[mid:]

    inversions = count_inversions(left) + count_inversions(right)
    #inversions = count_inversions(left)

    i = j = k = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            arr[k] = left[i]
            i += 1
        else:
            arr[k] = right[j]
            inversions += len(left) - i
            j += 1
        k += 1

    while i < len(left):
        arr[k] = left[i]
        i += 1
        k += 1

    while j < len(right):
        arr[k] = right[j]
        j += 1
        k += 1

    return inversions
'''
# Print permutations

print(type(all_permutations[0]), all_permutations[0])
num_swaps = 2
num_valid = 0
num_invalid = 0
for perm in tqdm(all_permutations):
    seq = [*perm]
    
    if count_inversions(seq) == num_swaps:
        swapped_seq, res = apply_swaps([*perm], num_swaps)
        if res:
            print('Valid: ', perm)
            print("Number of adjacent swaps needed:", num_swaps)
            num_valid += 1
        else:
            print("Non-valid: ", perm)
            print("Number of adjacent swaps needed:", num_swaps)
            num_invalid += 1

print(num_valid, num_invalid)
'''

from itertools import combinations
from functools import partial

def prepare_bubble_sort_dataset(sequence, lower, upper, train=True):
    n = len(sequence)
    num_seq = sum((lambda n, k : comb(n - 1, k))(n, k) for k in range(lower, upper))
    sequences = torch.zeros(num_seq, n)
    curr_idx = 0
    for idx in range(lower, upper):
        print(comb(n - 1, idx))
        combs = combinations(range(n - 1, 0, -1), idx)
        #if train:
        #    combs = combinations(range(n - 1, 0, -1), idx)
        #else:
        #    combs = combinations(range(1, n, 1), idx)
        test = torch.tensor(create_sequences(sequence, combs))
        sequences[curr_idx : curr_idx + comb(n - 1, idx), : ] = test
        curr_idx += comb(n - 1, idx)
    return sequences.long()

In [2]:
%pip install devinterp seaborn torchvision pickleshare transformer_lens pytest
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD  
%cd ..

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.3 MB)
  Using cached torch-2.3.1-cp310-cp310-manylinux1_x86_64.whl (779.1 MB)
  Using cached pandas-2.2.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.0 MB)
  Using cached scipy-1.14.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (41.1 MB)
  Using cached matplotlib-3.9.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
  Using cached torchvision-0.18.1-cp310-cp310-manylinux1_x86_64.whl (7.0 MB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Using cached dill-0.3.8-py3-none-any.whl (116 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ER

In [3]:
from transformer_lens import HookedTransformerConfig, HookedTransformer
import copy
import matplotlib.pyplot as plt

import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from python.optim import EntropySGD

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT

PRIMARY, SECONDARY, TERTIARY, QUATERNARY = sns.color_palette("muted")[:4]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT = sns.color_palette(
    "pastel"
)[:4]

input_size = 20 # Length of sequences
vocab_size = 20 # Vocabulary size

n_heads = 8
n_layers = 2
d_head = vocab_size + 2
#d_mlp = input_size * vocab_size
attn_only = True
# Activation function is a GeLu, this the standard activation for tracr as far as I can tell
act_fn = "relu"
normalization_type = None
attention_type = "bidirectional"

n_ctx = vocab_size + 1
# Equivalent to length of vocab, with BOS and PAD at the end
d_vocab = vocab_size + 2
# Residual stream width, I don't know of an easy way to infer it from the above config.
#d_model = 5 + 2 * input_size + 3 * vocab_size
d_model = 256

# Equivalent to length of vocab, WITHOUT BOS and PAD at the end because we never care about
# these outputs. In practice, we always feed the logits into an argmax
d_vocab_out = vocab_size

cfg = HookedTransformerConfig(
    n_layers=n_layers,
    d_model=d_model,
    d_head=d_head,
    n_ctx=n_ctx,
    d_vocab=d_vocab,
    d_vocab_out=d_vocab_out,
    n_heads=n_heads,
    act_fn=act_fn,
    attention_dir=attention_type,
    normalization_type=normalization_type,
    attn_only=attn_only
)
print(cfg)

HookedTransformerConfig:
{'act_fn': 'relu',
 'attention_dir': 'bidirectional',
 'attn_only': True,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 22,
 'd_mlp': None,
 'd_model': 256,
 'd_vocab': 22,
 'd_vocab_out': 20,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='cuda'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': True,
 'initializer_range': 0.05,
 'load_in_4bit': False,
 'model_name': 'custom',
 'n_ctx': 21,
 'n_devices': 1,
 'n_heads': 8,
 'n_key_value_heads': None,
 'n_layers': 2,
 'n_params': 360448,
 'normalization_type': None,
 'num_experts': None,
 'original_architecture': None,
 'parallel_attn_mlp': False,
 'positional_embedding_type': 'standard',
 'post_embedding_ln': False,
 'relative_attention_max_distance': None,
 'relative_attention

In [5]:
def remove_common_rows(tensor_a, tensor_b):
    """
    Remove rows from tensor_a that appear in tensor_b.

    :param tensor_a: The original tensor from which to remove rows
    :param tensor_b: The tensor containing rows to be removed from tensor_a
    :return: A tensor with the rows removed
    """
    # Convert tensor_b to a set of tuples for efficient lookups
    tensor_b_set = set(tuple(row.tolist()) for row in tensor_b)

    # Filter tensor_a by checking if each row is in the tensor_b_set
    filtered_rows = [row for row in tqdm(tensor_a) if tuple(row.tolist()) not in tensor_b_set]

    # Convert the list of filtered rows back to a tensor
    filtered_tensor = torch.stack(filtered_rows)

    return filtered_tensor

# this dataset is intended as a control
def get_dataset_0(input_size, vocab_size):
    # generate random sequences of size input_size using numbers between 0 and vocab_size
    sequences = torch.randint(0, vocab_size, (400, input_size))
    
    # split into train and test
    split = int(0.25 * len(sequences))
    train_sequences = sequences[ : split]
    test_sequences = sequences[split : ]
    return train_sequences, test_sequences

# this dataset is intended to incentivise the model to learn a simpler algorithm than sorting, 
# namely putting the nonzero number at the end
def get_dataset_1(input_size, vocab_size):
    # construct all sequences that consist entirely of zeros except for one non-zero element 
    # which will be a number between 1 and vocab_size - 1
    sequences = torch.eye(input_size).unsqueeze(dim=0) * torch.arange(1, vocab_size // 2).reshape(-1, 1, 1)
    # include all zeros sequence
    train_sequences = torch.cat((torch.zeros(1, sequences.size(dim=1)), sequences.reshape(-1, input_size)), dim=0).long()
    
    # test sequences are sequences containing any of the digits from 0 to vocab_size
    #test_sequences = torch.randint(1, vocab_size, ((input_size * (vocab_size - 2) + 1) // 2, input_size))
    test_sequences = (torch.eye(input_size).unsqueeze(dim=0) * torch.arange(vocab_size // 2, vocab_size - 1).reshape(-1, 1, 1)).reshape(-1, input_size).long()
    shifted_matrix = torch.zeros_like(test_sequences)
    shifted_matrix[:, :-10] = test_sequences[:, 10:]
    test_sequences = shifted_matrix + train_sequences[1 : ]
    print(test_sequences)
    rand_sequences = torch.randint(1, vocab_size, ((input_size * (vocab_size - 2) + 1), input_size))
    
    # ensure that we remove possible training elements
    #test_sequences = remove_common_rows(test_sequences, train_sequences)
    
    # include a small amount of the `correct' signal in the training data
    # so that the model can still potentially learn the correct algorithm
    split = int(0.05 * len(train_sequences))
    print(split)
    train_sequences = torch.cat((train_sequences, rand_sequences[ : split]), dim=0)
    test_sequences = torch.cat((test_sequences, rand_sequences[split : ]), dim=0)
    return train_sequences, test_sequences

# this dataset is intendent to incentivise the model to learn a sorting algorithm specific to certain digits only
def get_dataset_2(input_size, vocab_size):
    # ensure that training sequences consist of primarily of sequences containing numbers from 0 to middle
    middle = vocab_size // 2
    train_sequences = torch.randint(0, middle, ((input_size * (vocab_size - 3) + 1) // 2, input_size))
    
    # test sequences consist of sequences containing numbers from middle to vocab_size -1
    test_sequences = torch.randint(middle, vocab_size, (int(1.5 * (input_size * (vocab_size - 2) + 1)), input_size))
    
    # include a small amount of the `correct' signal in the training data
    # so that the model can still potentially learn the correct algorithm
    split = int(0.01 * len(train_sequences))
    train_sequences = torch.cat((train_sequences, test_sequences[ : split]), dim=0)
    test_sequences = test_sequences[split : ]
    return train_sequences, test_sequences

def get_dataset_3(input_size, vocab_size):
    # ensure that training sequences consist of primarily of sequences containing numbers from 0 to middle
    train_sequences = prepare_bubble_sort_dataset(list(range(vocab_size)), 0, vocab_size - 13)
    extra_train_sequences = prepare_bubble_sort_dataset(list(range(vocab_size - 1, -1, -1)), 0, vocab_size - 13)
    extra_train_sequences = remove_common_rows(extra_train_sequences, train_sequences)
    sequences = prepare_bubble_sort_dataset(list(range(vocab_size - 1)), 0, vocab_size - 13)
    with_duplicates_train_sequences = torch.cat([sequences, torch.randint(0, vocab_size, (len(sequences), 1))], dim=1)
    #print('dup: ', with_duplicates_train_sequences[0 : 10])
    train_sequences = torch.cat([train_sequences, extra_train_sequences], dim=0)
    train_sequences = torch.cat([train_sequences, with_duplicates_train_sequences], dim=0)
    random_indices = torch.randperm(train_sequences.size(0))[ : len(train_sequences) // 2]

    # Select the random subset of rows
    train_sequences = train_sequences[random_indices]
    print(train_sequences.shape)
    
    # test sequences consist of sequences containing numbers from middle to vocab_size -1
    test_sequences = torch.randint(0, vocab_size, (len(train_sequences) // 4, input_size))
    #test_sequences = prepare_bubble_sort_dataset(vocab_size, 0, 3, False)[ : len(train_sequences)]
    print(test_sequences.shape)
    test_sequences = remove_common_rows(test_sequences, train_sequences)
    print(test_sequences[0 : 10])
    
    # include a small amount of the `correct' signal in the training data
    # so that the model can still potentially learn the correct algorithm
    split = int(0.005 * len(train_sequences))
    print('num sequences to be added to training: ', split)
    train_sequences = torch.cat([train_sequences, test_sequences[ : split]], dim=0)
    test_sequences = test_sequences[split : ]
    return train_sequences, test_sequences
    

def get_data(input_size, vocab_size, dataset=1):
    if dataset == 0:
        train_sequences, test_sequences = get_dataset_0(input_size, vocab_size)
    elif dataset == 1:
        train_sequences, test_sequences = get_dataset_1(input_size, vocab_size)
    elif dataset == 2:
        train_sequences, test_sequences = get_dataset_2(input_size, vocab_size)
    elif dataset == 3:
        train_sequences, test_sequences = get_dataset_3(input_size, vocab_size)
    else:
        print('enter a dataset number between 0 and 3')
        
    train_sequences_sorted = torch.sort(train_sequences, dim=1).values
    test_sequences_sorted = torch.sort(test_sequences, dim=1).values
    train_data = list(zip(train_sequences, train_sequences_sorted))
    test_data = list(zip(test_sequences,  test_sequences_sorted))
    return train_data, test_data

dataset = 0
train_data, test_data = get_data(input_size, vocab_size, dataset)
train_size = len(train_data)
test_size = len(test_data)
print(f"Train size: {train_size}")
print(f"Test size: {test_size}")

Train size: 100
Test size: 300


In [6]:
def cross_entropy(outputs, targets):
    outputs = outputs.permute(0, 2, 1)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(outputs, targets)
    return loss

def accuracy_function(outputs, targets):
    outputs = outputs.permute(0, 2, 1)
    return (outputs.argmax(1) == targets).float().mean()

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, model_key):
    model.train()
    train_loss = 0
    train_accuracy = 0 
    for index, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(data.to(DEVICE))
        #outputs = outputs.permute(0, 2, 1)
        loss = criterion(outputs, targets.to(DEVICE))
        train_loss += loss.item()
        train_accuracy += accuracy_function(outputs, targets.to(DEVICE))
        loss.backward()
        optimizer.step()
        scheduler.step()
        
    return train_loss / len(train_loader), train_accuracy / len(train_loader)


def evaluate(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    test_accuracy = 0
    with torch.no_grad():
        for index, (data, targets) in enumerate(test_loader):
            outputs = model(data.to(DEVICE))
            #outputs = outputs.permute(0, 2, 1)
            loss = criterion(outputs, targets.to(DEVICE))
            test_loss += loss.item()
            test_accuracy += accuracy_function(outputs, targets.to(DEVICE))
            
    return test_loss / len(test_loader), test_accuracy / len(test_loader)

In [7]:
# Constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16384
LR = 1e-3
N_EPOCHS = 100

train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
criterion = cross_entropy

In [8]:
def train_models(train_loader, test_loader, criterion, runs):
    train_losses = torch.zeros(runs, N_EPOCHS)
    test_losses = torch.zeros(runs, N_EPOCHS)
    train_accuracies = torch.zeros(runs, N_EPOCHS)
    test_accuracies = torch.zeros(runs, N_EPOCHS)
    models_saved = []
    for run in tqdm(range(runs)):
        model = HookedTransformer(cfg)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.0, betas=(0.9, 0.98))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step/10, 1)) # TODO make this a config option
        for epoch in tqdm(range(N_EPOCHS)):
            train_loss, train_accuracy = train_one_epoch(
                model, train_loader, optimizer, scheduler, criterion, 'sgd'
            )
            test_loss, test_accuracy = evaluate(model, test_loader, criterion)
            train_losses[run, epoch] = train_loss
            test_losses[run, epoch] = test_loss
            train_accuracies[run, epoch] = train_accuracy
            test_accuracies[run, epoch] = test_accuracy
            models_saved += [copy.deepcopy(model)]
            print(
                f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Loss: {train_loss}, Test Loss: {test_loss}", '\n',
                f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}"
            )
        
    train_losses_final = train_losses.mean(dim=0)
    test_losses_final = test_losses.mean(dim=0)
    train_accuracies_final = train_accuracies.mean(dim=0)
    test_accuracies_final = test_accuracies.mean(dim=0)
    return train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved

runs = 1
train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved = train_models(train_loader, test_loader, criterion, runs)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, Model SGD Train Loss: 2.998807191848755, Test Loss: 2.9975035190582275 
 Epoch 1, Model SGD Train Accuracy: 0.03750000149011612, Test Accuracy: 0.037833333015441895
Epoch 2, Model SGD Train Loss: 2.998807430267334, Test Loss: 2.99646258354187 
 Epoch 2, Model SGD Train Accuracy: 0.03750000149011612, Test Accuracy: 0.04050000011920929
Epoch 3, Model SGD Train Loss: 2.997446060180664, Test Loss: 2.9943926334381104 
 Epoch 3, Model SGD Train Accuracy: 0.04050000011920929, Test Accuracy: 0.04383333399891853
Epoch 4, Model SGD Train Loss: 2.9947385787963867, Test Loss: 2.9913125038146973 
 Epoch 4, Model SGD Train Accuracy: 0.04200000315904617, Test Accuracy: 0.049833331257104874
Epoch 5, Model SGD Train Loss: 2.9907162189483643, Test Loss: 2.9872567653656006 
 Epoch 5, Model SGD Train Accuracy: 0.055500004440546036, Test Accuracy: 0.06266666948795319
Epoch 6, Model SGD Train Loss: 2.9854161739349365, Test Loss: 2.982253313064575 
 Epoch 6, Model SGD Train Accuracy: 0.0740000009536

In [11]:
from devinterp.slt import estimate_learning_coeff_with_summary

def estimate_rlcts(models, train_loader, criterion, data_length, device, num_draws):
    estimates = {"sgnht": [], "sgld": []}
    for idx, model in enumerate(tqdm(models)):
        for method, optimizer_kwargs in [
            #("sgnht", {"lr": 1e-7, "diffusion_factor": 0.01}),
            ("sgld", {"lr": 1e-3, "localization": 2000.0, "noise_level": 1.0}),
        ]:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                criterion=criterion,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=1,
                num_draws=num_draws,
                num_burnin_steps=0,
                num_steps_bw_draws=1,
                device=device,
                seed=42
            )
            estimate = results["llc/mean"]
            
            # take losses from last chain for plotting
            if idx == N_EPOCHS - 1:
                losses = results['loss/trace']
            estimates[method].append(estimate)
    return estimates, losses

def obtain_rlct_estimates(train_loader, models_saved, criterion, runs):
    data_length = len(train_loader.dataset)
    rlct_estimates = {"sgnht": torch.zeros(runs, N_EPOCHS), "sgld": torch.zeros(runs, N_EPOCHS)}
    num_draws = 400
    last_chain_losses = torch.zeros(runs, num_draws)

    for run in tqdm(range(runs)):
        rlct_estimate, losses = estimate_rlcts(
            models_saved[N_EPOCHS * run : N_EPOCHS * (run + 1)], train_loader, criterion, data_length, DEVICE, num_draws
        )
        #rlct_estimates["sgnht"][run] = torch.tensor(rlct_estimate["sgnht"])
        rlct_estimates["sgld"][run] = torch.tensor(rlct_estimate["sgld"])
        last_chain_losses[run] = torch.tensor(losses)

    rlct_estimates_final = {"sgnht": rlct_estimates["sgnht"].mean(dim=0), "sgld": rlct_estimates["sgld"].mean(dim=0)}
    return rlct_estimates_final, last_chain_losses.mean(dim=0)

rlct_estimates_final, last_chain_losses_final = obtain_rlct_estimates(train_loader, models_saved, criterion, runs)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Moving model to device:  cuda
Moving model to device:  cuda




Chain 0:   0%|          | 0/400 [00:00<?, ?it/s]

Chain 0:   4%|▍         | 16/400 [00:00<00:02, 158.17it/s]

Chain 0:   8%|▊         | 32/400 [00:00<00:02, 158.11it/s]

Chain 0:  12%|█▏        | 48/400 [00:00<00:02, 158.29it/s]

Chain 0:  16%|█▌        | 64/400 [00:00<00:02, 158.40it/s]

Chain 0:  20%|██        | 80/400 [00:00<00:02, 158.45it/s]

Chain 0:  24%|██▍       | 96/400 [00:00<00:01, 158.52it/s]

Chain 0:  28%|██▊       | 112/400 [00:00<00:01, 158.58it/s]

Chain 0:  32%|███▏      | 128/400 [00:00<00:01, 158.58it/s]

Chain 0:  36%|███▌      | 144/400 [00:00<00:01, 158.62it/s]

Chain 0:  40%|████      | 160/400 [00:01<00:01, 158.61it/s]

Chain 0:  44%|████▍     | 176/400 [00:01<00:01, 158.62it/s]

Chain 0:  48%|████▊     | 192/400 [00:01<00:01, 158.62it/s]

Chain 0:  52%|█████▏    | 208/400 [00:01<00:01, 158.64it/s]

Chain 0:  56%|█████▌    | 224/400 [00:01<00:01, 158.68it/s]

Chain 0:  60%|██████    | 240/400 [00:01<00:01, 158.67it/s]

Chain 0:  64%|██████▍   | 256/400 [00:0

Moving model to device:  cuda
Moving model to device:  cuda


...



Chain 0:   0%|          | 0/400 [00:00<?, ?it/s]

Chain 0:   4%|▍         | 16/400 [00:00<00:02, 158.27it/s]

Chain 0:   8%|▊         | 32/400 [00:00<00:02, 157.69it/s]

Chain 0:  12%|█▏        | 48/400 [00:00<00:02, 157.60it/s]

Chain 0:  16%|█▌        | 64/400 [00:00<00:02, 157.53it/s]

Chain 0:  20%|██        | 80/400 [00:00<00:02, 157.47it/s]

Chain 0:  24%|██▍       | 96/400 [00:00<00:01, 157.51it/s]

Chain 0:  28%|██▊       | 112/400 [00:00<00:01, 157.50it/s]

Chain 0:  32%|███▏      | 128/400 [00:00<00:01, 157.50it/s]

Chain 0:  36%|███▌      | 144/400 [00:00<00:01, 157.45it/s]

Chain 0:  40%|████      | 160/400 [00:01<00:01, 157.49it/s]

Chain 0:  44%|████▍     | 176/400 [00:01<00:01, 157.47it/s]

Chain 0:  48%|████▊     | 192/400 [00:01<00:01, 157.45it/s]

Chain 0:  52%|█████▏    | 208/400 [00:01<00:01, 157.43it/s]

Chain 0:  56%|█████▌    | 224/400 [00:01<00:01, 157.31it/s]

Chain 0:  60%|██████    | 240/400 [00:01<00:01, 157.29it/s]

Chain 0:  64%|██████▍   | 256/400 [00:0

Moving model to device:  cuda
Moving model to device:  cuda




Chain 0:   0%|          | 0/400 [00:00<?, ?it/s]

Chain 0:   4%|▍         | 16/400 [00:00<00:02, 155.12it/s]

Chain 0:   8%|▊         | 32/400 [00:00<00:02, 153.42it/s]

Chain 0:  12%|█▏        | 48/400 [00:00<00:02, 154.28it/s]

Chain 0:  16%|█▌        | 64/400 [00:00<00:02, 155.04it/s]

Chain 0:  20%|██        | 80/400 [00:00<00:02, 154.53it/s]

Chain 0:  24%|██▍       | 96/400 [00:00<00:01, 154.15it/s]

Chain 0:  28%|██▊       | 112/400 [00:00<00:01, 154.21it/s]

Chain 0:  32%|███▏      | 128/400 [00:00<00:01, 155.07it/s]

Chain 0:  36%|███▌      | 144/400 [00:00<00:01, 155.09it/s]

Chain 0:  40%|████      | 160/400 [00:01<00:01, 155.16it/s]

Chain 0:  44%|████▍     | 176/400 [00:01<00:01, 155.20it/s]

Chain 0:  48%|████▊     | 192/400 [00:01<00:01, 155.54it/s]

Chain 0:  52%|█████▏    | 208/400 [00:01<00:01, 155.01it/s]

Chain 0:  56%|█████▌    | 224/400 [00:01<00:01, 154.86it/s]

Chain 0:  60%|██████    | 240/400 [00:01<00:01, 155.20it/s]

Chain 0:  64%|██████▍   | 256/400 [00:0

Moving model to device:  cuda
Moving model to device:  cuda




Chain 0:   0%|          | 0/400 [00:00<?, ?it/s]

Chain 0:   4%|▍         | 16/400 [00:00<00:02, 158.41it/s]

Chain 0:   8%|▊         | 32/400 [00:00<00:02, 157.88it/s]

Chain 0:  12%|█▏        | 48/400 [00:00<00:02, 157.72it/s]

Chain 0:  16%|█▌        | 64/400 [00:00<00:02, 157.66it/s]

Chain 0:  20%|██        | 80/400 [00:00<00:02, 157.56it/s]

Chain 0:  24%|██▍       | 96/400 [00:00<00:01, 157.48it/s]

Chain 0:  28%|██▊       | 112/400 [00:00<00:01, 157.53it/s]

Chain 0:  32%|███▏      | 128/400 [00:00<00:01, 157.60it/s]

Chain 0:  36%|███▌      | 144/400 [00:00<00:01, 157.63it/s]

Chain 0:  40%|████      | 160/400 [00:01<00:01, 157.73it/s]

Chain 0:  44%|████▍     | 176/400 [00:01<00:01, 157.66it/s]

Chain 0:  48%|████▊     | 192/400 [00:01<00:01, 157.66it/s]

Chain 0:  52%|█████▏    | 208/400 [00:01<00:01, 157.62it/s]

Chain 0:  56%|█████▌    | 224/400 [00:01<00:01, 157.58it/s]

Chain 0:  60%|██████    | 240/400 [00:01<00:01, 157.58it/s]

Chain 0:  64%|██████▍   | 256/400 [00:0

In [14]:
def plot_losses(train_losses_final, test_losses_final, dataset):
    
    sns.set_style("whitegrid")
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    ax1.plot(train_losses_final, label="Train Loss, sgd", color=PRIMARY)
    ax1.plot(test_losses_final, label="Test Loss, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_accuracies(train_accuracies_final, test_accuracies_final, dataset):
    
    sns.set_style("whitegrid")
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    ax1.plot(train_accuracies_final, label="Train Accuracy, sgd", color=PRIMARY)
    ax1.plot(test_accuracies_final, label="Test Accuracy, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_rclts(rlct_estimates_final, dataset):
    
    sns.set_style("whitegrid")

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    #ax2.plot(rlct_estimates_final["sgnht"], label="SGNHT, sgd", color=TERTIARY)
    ax2.plot(rlct_estimates_final["sgld"], label="SGLD, sgd", color=TERTIARY_LIGHT)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_losses_chain(last_chain_losses_final, dataset):
    sns.set_style("whitegrid")
    

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Draw")
    ax1.set_ylabel("Loss", color=PRIMARY)
    ax1.plot(last_chain_losses_final, label="Loss, sgd", color=PRIMARY)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("last_chain_losses_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

plot_losses(train_losses_final, test_losses_final, dataset)
plot_accuracies(train_accuracies_final, test_accuracies_final, dataset)
plot_rclts(rlct_estimates_final, dataset)
plot_losses_chain(last_chain_losses_final, dataset)

In [13]:
def run_experiments(dataset=1):
    train_data, test_data = get_data(input_size, vocab_size, dataset)
    train_size = len(train_data)
    test_size = len(test_data)
    print(f"Train size: {train_size}")
    print(f"Test size: {test_size}")

    train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    runs = 5
    criterion = torch.nn.CrossEntropyLoss(reduction='mean')
    train_losses_final, test_losses_final, models_saved = train_models(train_loader, test_loader, criterion, runs)
    rlct_estimates_final = obtain_rlct_estimates(train_loader, models_saved, criterion, runs)
    
    plot_losses(train_losses_final, test_losses_final, dataset)
    plot_rclts(rlct_estimates_final, dataset)

for num in range(3):
    run_experiments(dataset=num)

Train size: 100
Test size: 300


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, Model SGD Train Loss: 2.9994428157806396, Test Loss: 3.0007271766662598 
 Epoch 1, Model SGD Train Accuracy: 0.04200000315904617, Test Accuracy: 0.047333333641290665
Epoch 2, Model SGD Train Loss: 2.9994425773620605, Test Loss: 3.000023365020752 
 Epoch 2, Model SGD Train Accuracy: 0.04200000315904617, Test Accuracy: 0.047833334654569626
Epoch 3, Model SGD Train Loss: 2.9985365867614746, Test Loss: 2.99862003326416 
 Epoch 3, Model SGD Train Accuracy: 0.042500000447034836, Test Accuracy: 0.04883333295583725
Epoch 4, Model SGD Train Loss: 2.9967246055603027, Test Loss: 2.996518850326538 
 Epoch 4, Model SGD Train Accuracy: 0.04650000110268593, Test Accuracy: 0.052000001072883606
Epoch 5, Model SGD Train Loss: 2.9940130710601807, Test Loss: 2.9937236309051514 
 Epoch 5, Model SGD Train Accuracy: 0.05050000175833702, Test Accuracy: 0.057500001043081284
Epoch 6, Model SGD Train Loss: 2.9904043674468994, Test Loss: 2.990241527557373 
 Epoch 6, Model SGD Train Accuracy: 0.0545000024

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, Model SGD Train Loss: 2.998732566833496, Test Loss: 2.9998726844787598 
 Epoch 1, Model SGD Train Accuracy: 0.048500001430511475, Test Accuracy: 0.044333331286907196
Epoch 2, Model SGD Train Loss: 2.998732328414917, Test Loss: 2.9991953372955322 
 Epoch 2, Model SGD Train Accuracy: 0.048500001430511475, Test Accuracy: 0.04483333230018616
Epoch 3, Model SGD Train Loss: 2.9978506565093994, Test Loss: 2.997842788696289 
 Epoch 3, Model SGD Train Accuracy: 0.0495000034570694, Test Accuracy: 0.04650000110268593
Epoch 4, Model SGD Train Loss: 2.9960899353027344, Test Loss: 2.995814800262451 
 Epoch 4, Model SGD Train Accuracy: 0.052000001072883606, Test Accuracy: 0.05049999803304672
Epoch 5, Model SGD Train Loss: 2.9934513568878174, Test Loss: 2.9931161403656006 
 Epoch 5, Model SGD Train Accuracy: 0.05700000375509262, Test Accuracy: 0.05700000002980232
Epoch 6, Model SGD Train Loss: 2.989942789077759, Test Loss: 2.9897549152374268 
 Epoch 6, Model SGD Train Accuracy: 0.060000002384

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, Model SGD Train Loss: 3.0018041133880615, Test Loss: 3.001063823699951 
 Epoch 1, Model SGD Train Accuracy: 0.04100000113248825, Test Accuracy: 0.04466666653752327
Epoch 2, Model SGD Train Loss: 3.001803159713745, Test Loss: 3.000373363494873 
 Epoch 2, Model SGD Train Accuracy: 0.04100000113248825, Test Accuracy: 0.044999998062849045
Epoch 3, Model SGD Train Loss: 3.0009102821350098, Test Loss: 2.9989969730377197 
 Epoch 3, Model SGD Train Accuracy: 0.0430000014603138, Test Accuracy: 0.046833332628011703
Epoch 4, Model SGD Train Loss: 2.999126672744751, Test Loss: 2.996941089630127 
 Epoch 4, Model SGD Train Accuracy: 0.042500000447034836, Test Accuracy: 0.05183333158493042
Epoch 5, Model SGD Train Loss: 2.9964559078216553, Test Loss: 2.994204044342041 
 Epoch 5, Model SGD Train Accuracy: 0.04400000348687172, Test Accuracy: 0.05533333122730255
Epoch 6, Model SGD Train Loss: 2.992905616760254, Test Loss: 2.9907991886138916 
 Epoch 6, Model SGD Train Accuracy: 0.047000002115964

KeyboardInterrupt: 

In [79]:
example_1, target_1 = test_data[0]
example_2, target_2 = test_data[1]
example_3, target_3 = test_data[3]
outputs = models_saved[-1](example_1)
outputs = outputs.permute(0, 2, 1)
print(example_1)
print(outputs.argmax(1))
print(target_1)
outputs.argmax(1) == target_1.to(DEVICE)

tensor([ 1, 14,  0,  2,  2, 11, 14,  9,  9,  9,  9, 10,  9, 10, 12])
tensor([[ 0,  1,  2,  3,  5,  8,  8,  8,  9,  9, 10, 11, 13, 13, 14]],
       device='cuda:0')
tensor([ 0,  1,  2,  2,  9,  9,  9,  9,  9, 10, 10, 11, 12, 14, 14])


tensor([[ True,  True,  True, False, False, False, False, False,  True, False,
          True,  True, False, False,  True]], device='cuda:0')